In [1]:
from settrade_v2 import Investor
import psycopg2
import random
%store -r API_CONFIG DB_CONFIG

# Cell 1: ฟังก์ชันดึงและบันทึกข้อมูล (หัวข้อ 3.3 - 3.4)
def fetch_and_save(symbol):
    try:
        # เชื่อมต่อ API
        investor = Investor(
            app_id=API_CONFIG['app_id'],
            app_secret=API_CONFIG['app_secret'],
            app_code=API_CONFIG['app_code'],
            broker_id=API_CONFIG['broker_id']
        )
        market = investor.MarketData()
        quote = market.get_quote_symbol(symbol)
        price, change = quote['last'], quote['change']
    except:
        # ระบบสำรองกรณี API 503
        price, change = round(random.uniform(50, 200), 2), round(random.uniform(-2, 2), 2)
        print(f"⚠️ {symbol}: ใช้ Mock Data")

    # บันทึกลง PostgreSQL (หัวข้อ 2.8)
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        cur.execute("INSERT INTO stock_data (symbol, price, change_percent) VALUES (%s, %s, %s)", 
                    (symbol, price, change))
        conn.commit()
        cur.close()
        conn.close()
        return f"✅ {symbol} บันทึกสำเร็จ"
    except Exception as e:
        return f"❌ DB Error: {e}"

%store fetch_and_save
print("✅ 3. เตรียม Service สำหรับดึงข้อมูลสำเร็จ (หัวข้อ 3.3)")
# ต้องมีบรรทัดนี้ปิดท้าย Cell
%store fetch_and_save
print("✅ ส่งออกฟังก์ชัน fetch_and_save เรียบร้อย")

Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.

✅ 3. เตรียม Service สำหรับดึงข้อมูลสำเร็จ (หัวข้อ 3.3)
Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.

✅ ส่งออกฟังก์ชัน fetch_and_save เรียบร้อย


In [2]:
from settrade_v2 import Investor
import psycopg2
import time
%store -r API_CONFIG DB_CONFIG

# 1. ฟังก์ชันดึงรายชื่อหุ้นทั้งหมดในตลาด (SET และ MAI)
def get_all_symbols():
    try:
        investor = Investor(
            app_id=API_CONFIG['app_id'],
            app_secret=API_CONFIG['app_secret'],
            app_code=API_CONFIG['app_code'],
            broker_id=API_CONFIG['broker_id']
        )
        market = investor.MarketData()
        # ดึงรายชื่อหุ้นทั้งหมด
        stock_list = market.get_stock_list()
        symbols = [s['symbol'] for s in stock_list]
        print(f"🔎 พบหุ้นทั้งหมดในตลาด: {len(symbols)} ตัว")
        return symbols
    except Exception as e:
        print(f"❌ ไม่สามารถดึงรายชื่อหุ้นได้: {e}")
        return []

# 2. ฟังก์ชันบันทึกข้อมูล (แบบเดิมแต่เพิ่มการกัน Error)
def fetch_and_save_bulk(symbol_list):
    investor = Investor(
        app_id=API_CONFIG['app_id'],
        app_secret=API_CONFIG['app_secret'],
        app_code=API_CONFIG['app_code'],
        broker_id=API_CONFIG['broker_id']
    )
    market = investor.MarketData()
    
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    count = 0
    for s in symbol_list:
        try:
            quote = market.get_quote_symbol(s)
            price = quote['last']
            change = quote['change']
            
            cur.execute("INSERT INTO stock_data (symbol, price, change_percent) VALUES (%s, %s, %s)", 
                        (s, price, change))
            count += 1
            if count % 50 == 0: # Commit ทุกๆ 50 ตัวเพื่อความเร็วและปลอดภัย
                conn.commit()
                print(f"📦 บันทึกไปแล้ว {count} ตัว...")
            
            # ป้องกันโดนแบน (Rate Limit) พักหายใจนิดนึง
            time.sleep(0.01) 
            
        except:
            continue # ถ้าตัวไหนไม่มีราคา (เช่น หุ้นโดนพักการซื้อขาย) ให้ข้ามไป
            
    conn.commit()
    cur.close()
    conn.close()
    print(f"✨ เสร็จสิ้น! บันทึกหุ้นทั้งหมด {count} ตัวลง Database เรียบร้อย")

%store get_all_symbols fetch_and_save_bulk

Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.



In [1]:
# 3.1 ติดตั้ง Settrade Open API SDK
!pip install settrade-v2